TASK 4 - EVALUATION OF FINETUNED MODELS ON PF-WILLOW

In [ ]:
# ============================================================================
# TASK 4: EVALUATE FINETUNED MODELS ON PF-WILLOW DATASET
# ============================================================================

# ====================
# SETUP
# ====================

# 1. Clone repositories
!git clone https://github.com/Luffy65/Semantic-Correspondence.git  # our repo

# uncomment git checkout if needed
# Switch to dev-task4 branch
# %cd Semantic-Correspondence
# !git checkout dev-task4
# %cd ..

!git clone https://github.com/facebookresearch/dinov2.git           # DINOv2
!git clone https://github.com/facebookresearch/dinov3.git           # DINOv3
!pip install git+https://github.com/facebookresearch/segment-anything.git  # SAM

Cloning into 'Semantic-Correspondence'...
remote: Enumerating objects: 380, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 380 (delta 102), reused 117 (delta 48), pack-reused 208 (from 1)
Receiving objects: 100% (380/380), 7.01 MiB | 22.51 MiB/s, done.
Resolving deltas: 100% (206/206), done.
Cloning into 'dinov2'...
remote: Enumerating objects: 708, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 708 (delta 24), reused 12 (delta 12), pack-reused 659 (from 4)
Receiving objects: 100% (708/708), 2.96 MiB | 13.00 MiB/s, done.
Resolving deltas: 100% (324/324), done.
Cloning into 'dinov3'...
remote: Enumerating objects: 538, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (263/263), done.
remote: Total 538 (delta 199), reused 99 (delta 99), pack-reused 176 (from 1)
Receiving objects: 100% (538/538), 9.88 MiB | 13.06

In [ ]:

# 2. Install requirements
!pip install -r Semantic-Correspondence/requirements.txt
!pip install -r dinov2/requirements.txt
!pip install -r dinov3/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu117, https://pypi.nvidia.com
ERROR: Could not find a version that satisfies the requirement torch==2.0.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==2.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 61.1 MB/s eta 0:00:00


In [ ]:
# ====================
# LIBRARIES
# ====================

import torch
import os
import sys
import cv2
import numpy as np
import json
from PIL import Image
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
from segment_anything import SamPredictor, sam_model_registry
import shutil
import itertools
import glob
!pip install scipy
import scipy.io as sio

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ====================
# EXTRACT PF-WILLOW DATASET FROM DRIVE
# ====================

# Paths
DRIVE_ROOT = '/content/drive/MyDrive/AML-PROJECT-DATA'
DATASET_ROOT = os.path.join(DRIVE_ROOT, 'dataset')
DATASET_ARCHIVE = os.path.join(DATASET_ROOT, 'pf-willow.zip')
LOCAL_DATA_DIR = '/content/data'

# Extract dataset to local VM
if not os.path.exists(LOCAL_DATA_DIR):
    print(f"Extracting {DATASET_ARCHIVE} to local VM...")
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    shutil.unpack_archive(DATASET_ARCHIVE, LOCAL_DATA_DIR, format='zip')
    print(f"Done! Data is ready at {LOCAL_DATA_DIR}")
else:
    print("Data already loaded.")

PFWILLOW_ROOT = LOCAL_DATA_DIR

Mounted at /content/drive
Extracting /content/drive/MyDrive/AML-PROJECT-DATA/dataset/pf-willow.zip to local VM...
Done! Data is ready at /content/data


In [ ]:
# ====================
# IMPORT PF-WILLOW DATASET CLASS
# ====================

# Add Semantic-Correspondence to path
sys.path.insert(0, '/content/Semantic-Correspondence/datasets')

# Import the PFWillowDataset class from our repo
from Pf_willow import PFWillowDataset

MODEL LOADING AND PATH CONFIG

In [ ]:
# ====================
# LOAD MODELS WITH FINETUNED WEIGHTS
# ====================

# Paths to our finetuned checkpoints in Google Drive
CHECKPOINT_ROOT = '/content/drive/MyDrive/AML-PROJECT-DATA/checkpoints/finetuned/final'

# Repository directories
DINOV2_REPO_DIR = 'dinov2'
DINOV3_REPO_DIR = 'dinov3'

CHOOSE ONE OF THE CELLS IN THIS SECTION TO SELECT A MODEL TO EVALUATE

DINOV2 - FINETUNED

In [ ]:
DINOV2_FINETUNED_PATH = os.path.join(CHECKPOINT_ROOT, 'dinov2_finetuned.pth')
# Load DINOv2 with finetuned weights
dinov2_vitb14 = torch.hub.load(DINOV2_REPO_DIR, 'dinov2_vitb14', source='local', pretrained=False)
dinov2_checkpoint = torch.load(DINOV2_FINETUNED_PATH, map_location='cuda',  weights_only=False)
dinov2_vitb14.load_state_dict(dinov2_checkpoint['model_state_dict'])
dinov2_vitb14.eval()
dinov2_vitb14.cuda()
print(f"✓ DINOv2 loaded from {DINOV2_FINETUNED_PATH}")

✓ DINOv2 loaded from /content/drive/MyDrive/AML-PROJECT-DATA/checkpoints/finetuned/final/dinov2_finetuned.pth


DINO V3- FINETUNED

In [ ]:
DINOV3_FINETUNED_PATH = os.path.join(CHECKPOINT_ROOT, 'dinov3_finetuned.pth')
# Load DINOv3 with finetuned weights
dinov3_vitb16 = torch.hub.load(DINOV3_REPO_DIR, 'dinov3_vitb16', source='local', pretrained=False)
dinov3_checkpoint = torch.load(DINOV3_FINETUNED_PATH, map_location='cuda', weights_only=False)
dinov3_vitb16.load_state_dict(dinov3_checkpoint['model_state_dict'])
dinov3_vitb16.eval()
dinov3_vitb16.cuda()
print(f"✓ DINOv3 loaded from {DINOV3_FINETUNED_PATH}")

SAM - FINETUNED

In [ ]:
SAM_FINETUNED_PATH = os.path.join(CHECKPOINT_ROOT, 'sam_finetuned.pth')
# Load SAM with finetuned weights
sam = sam_model_registry['vit_b']()
sam_predictor = SamPredictor(sam)
sam_checkpoint = torch.load(SAM_FINETUNED_PATH, map_location='cuda', weights_only=False)
sam_predictor.model.image_encoder.load_state_dict(sam_checkpoint['model_state_dict'])
sam_predictor.model.cuda()
sam_predictor.model.eval()
print(f"✓ SAM loaded from {SAM_FINETUNED_PATH}")

FEATURE EXTRACTION

In [ ]:
# ====================
# FEATURE EXTRACTION FUNCTIONS
# ====================

def extract_dino_features(model, img_tensor):
    """
    Extracts dense features from DINO-like models (ViT).
    Returns: (1, Feature_Dim, H_grid, W_grid)
    """
    model.eval()
    with torch.no_grad():
        if hasattr(model, 'forward_features'):
            out = model.forward_features(img_tensor)
            # Handle dictionary output (common in DINOv2/v3)
            if isinstance(out, dict):
                patch_tokens = out.get("x_norm_patchtokens", out.get("x_norm_patch_tokens"))
            else:
                patch_tokens = out

            if patch_tokens is None:
                raise ValueError(f"Could not find patch tokens. Keys: {out.keys() if isinstance(out, dict) else 'N/A'}")

            # Reshape: (B, N, D) -> (B, D, H, W)
            B, N, D = patch_tokens.shape
            grid_size = int(np.sqrt(N))
            feature_map = patch_tokens.permute(0, 2, 1).reshape(B, D, grid_size, grid_size)
            return feature_map
    return None

def extract_dino_layers(model, img_tensor, layer_ids):
    """
    Extracts features from intermediate DINO layers.
    layer_ids: list of transformer block indices you want (0-based).
    returns dict: {layer_id: feature_map}
    """
    model.eval()
    with torch.no_grad():

        all_layers_outputs = model.get_intermediate_layers(img_tensor, n=len(model.blocks), norm=True)
        out = {}
        for lid in layer_ids:
            patch_tokens = all_layers_outputs[lid]
            B, N_patches, D = patch_tokens.shape

            # Dynamically calculate grid dimensions based on the number of patch tokens
            grid_size = int(np.sqrt(N_patches))
            H_grid = grid_size
            W_grid = grid_size

            if N_patches != H_grid * W_grid:
                raise ValueError(
                    f"Unexpected number of patch tokens for layer {lid}: {N_patches}. "
                    f"N_patches ({N_patches}) is not a perfect square for a grid. "
                    f"Calculated grid size: {H_grid}x{W_grid} = {H_grid*W_grid}."
                )

            # Reshape: (B, N_patches, D) -> (B, D, H_grid, W_grid)
            feature_map = patch_tokens.permute(0, 2, 1).reshape(B, D, H_grid, W_grid)
            out[lid] = feature_map
        return out

def extract_sam_features(predictor, image_np, res=512, layer_idx=None):
    """
    Extract features from SAM's image encoder with support for variable input resolutions.

    SAM was trained with 1024x1024 images (64x64 patches). For smaller resolutions,
    we temporarily swap in interpolated positional embeddings, then restore them.
    """
    device = next(predictor.model.parameters()).device
    encoder = predictor.model.image_encoder

    # Resize to specified resolution
    image_resized = cv2.resize(image_np, (res, res))

    # Convert to tensor
    img_tensor = torch.from_numpy(image_resized).float().permute(2, 0, 1).unsqueeze(0).to(device)

    # Apply SAM's preprocessing
    pixel_mean = torch.tensor([123.675, 116.28, 103.53]).view(-1, 1, 1).to(device)
    pixel_std = torch.tensor([58.395, 57.12, 57.375]).view(-1, 1, 1).to(device)
    img_tensor = (img_tensor - pixel_mean) / pixel_std

    with torch.no_grad():
        # Check if we need to interpolate positional embeddings
        orig_pos_embed = encoder.pos_embed
        new_size = res // 16  # SAM uses patch size of 16

        if orig_pos_embed is not None and orig_pos_embed.shape[1] != new_size:
            # Interpolate: (1, 64, 64, C) -> (1, C, 64, 64) -> interpolate -> (1, new, new, C)
            pos_embed_interp = F.interpolate(
                orig_pos_embed.data.permute(0, 3, 1, 2),
                size=(new_size, new_size),
                mode='bicubic',
                align_corners=False
            ).permute(0, 2, 3, 1)

            # Temporarily swap positional embeddings (wrap as Parameter)
            encoder.pos_embed = torch.nn.Parameter(pos_embed_interp, requires_grad=False)
            features = encoder(img_tensor)
            encoder.pos_embed = orig_pos_embed  # Restore original
        else:
            features = encoder(img_tensor)

    return features

In [18]:
# ====================
# COMPUTE PCK - PF WILLOW
# ====================

def computePCKatT_PFWillow(model, dataset_root, layer_id=None, thresholds=[0.05, 0.1, 0.15, 0.2],
                           img_size=(1024, 1024), use_softargmax=False,
                           softargmax_temp=0.01, softargmax_window=5):
    """
    Compute PCK (Percentage of Correct Keypoints) at various thresholds for PF-Willow.
    Adapted from the computePCKatT method.

    Key difference from SPair-71k: Uses image diagonal for normalization instead of bbox max(w,h).

    Returns:
        - per_keypoint: PCK computed as (total correct keypoints) / (total keypoints)
        - per_image: PCK computed for each image pair
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    is_sam = False
    if "SamPredictor" in str(type(model)):
        is_sam = True
    else:
        model = model.to(device)

    # PF-Willow dataset structure - AUTO-DISCOVER CATEGORIES
    pf_dataset_dir = os.path.join(dataset_root, 'PF-dataset')

    if not os.path.exists(pf_dataset_dir):
        print(f"ERROR: {pf_dataset_dir} does not exist!")
        return {'per_keypoint': {'overall': {f'PCK@{t}': 0.0 for t in thresholds}, 'per_category': {}},
                'per_image': [], 'total_keypoints': 0, 'total_images': 0}

    # Auto-discover category directories (exclude hidden/system dirs)
    all_items = os.listdir(pf_dataset_dir)
    categories = [d for d in all_items
                  if os.path.isdir(os.path.join(pf_dataset_dir, d))
                  and not d.startswith('.')
                  and not d.startswith('_')
                  and d != '__MACOSX']

    if not categories:
        print(f"ERROR: No category directories found in {pf_dataset_dir}")
        print(f"Items in directory: {all_items}")
        return {'per_keypoint': {'overall': {f'PCK@{t}': 0.0 for t in thresholds}, 'per_category': {}},
                'per_image': [], 'total_keypoints': 0, 'total_images': 0}

    print(f"Auto-discovered {len(categories)} categories: {categories}")

    # Generate image pairs
    pair_list = []
    print("Generating PF-Willow image pairs...")

    for category in categories:
        cat_dir = os.path.join(pf_dataset_dir, category)

        # Find all images with annotations
        img_files = glob.glob(os.path.join(cat_dir, '*.png'))

        img_names = []
        for img_path in img_files:
            img_name = os.path.splitext(os.path.basename(img_path))[0]
            if os.path.exists(os.path.join(cat_dir, f"{img_name}.mat")):
                img_names.append(img_name)

        category_pairs = list(itertools.permutations(img_names, 2)) # permutations gives 900 image pairs, combinations 450

        for src, trg in category_pairs:
            pair_list.append((category, src, trg))

        print(f"  Category '{category}': {len(img_names)} images -> {len(category_pairs)} pairs")

    print(f"Generated total of {len(pair_list)} image pairs across {len(categories)} categories")

    if len(pair_list) == 0:
        print(f"\nERROR: No image pairs found!")
        return {
            'per_keypoint': {'overall': {f'PCK@{t}': 0.0 for t in thresholds}, 'per_category': {}},
            'per_image': [],
            'total_keypoints': 0,
            'total_images': 0
        }

    # Transform for DINO
    transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Helper function to load keypoints from .mat files
    def load_keypoints(anno_path_base, cat_dir, img_name):
        """Load keypoints from .mat"""
        mat_path = os.path.join(cat_dir, f"{img_name}.mat")
        if os.path.exists(mat_path):
            try:
                mat = sio.loadmat(mat_path)
                # Try different possible keys
                for key in ["pts_coord", "pts", "kps", "keypoints"]:
                    if key in mat:
                        arr = mat[key]
                        arr = np.squeeze(arr)
                        if arr.ndim == 2 and arr.shape[0] == 2:
                            xy = arr.T
                        elif arr.ndim == 2 and arr.shape[1] == 2:
                            xy = arr
                        else:
                            continue
                        vis = np.ones((xy.shape[0], 1), dtype=np.float32)
                        kps_array = np.concatenate([xy.astype(np.float32), vis], axis=1)
                        return kps_array.tolist()
            except Exception as e:
                print(f"Warning: Error loading .mat file {mat_path}: {e}")
        return []

    # Per-keypoint tracking
    correct_kps = {t: 0 for t in thresholds}
    total_kps = 0

    # Per-category tracking
    category_correct = {t: {} for t in thresholds}
    category_total = {}

    # Per-image tracking
    per_image_results = []

    # Progress tracking
    num_pairs_processed = 0

    for category, src_name, trg_name in tqdm(pair_list, desc="Evaluating pairs"):
        if category not in category_total:
            category_total[category] = 0
            for t in thresholds:
                category_correct[t][category] = 0

        cat_dir = os.path.join(pf_dataset_dir, category)

        # Load images
        src_img_path = os.path.join(cat_dir, f"{src_name}.png")
        trg_img_path = os.path.join(cat_dir, f"{trg_name}.png")

        if not os.path.exists(src_img_path) or not os.path.exists(trg_img_path):
            continue

        src_pil = Image.open(src_img_path).convert('RGB')
        trg_pil = Image.open(trg_img_path).convert('RGB')
        src_w, src_h = src_pil.size
        trg_w, trg_h = trg_pil.size

        # Load keypoints using helper function
        src_kps = load_keypoints(None, cat_dir, src_name)
        trg_kps = load_keypoints(None, cat_dir, trg_name)

        if len(src_kps) == 0 or len(trg_kps) == 0:
            continue

        # Extract Features
        if is_sam:
            f_src = extract_sam_features(model, np.array(src_pil), res=img_size[0], layer_idx=layer_id)
            f_trg = extract_sam_features(model, np.array(trg_pil), res=img_size[0], layer_idx=layer_id)
        else:
            # DINO logic
            src_tensor = transform(src_pil).unsqueeze(0).to(device)
            trg_tensor = transform(trg_pil).unsqueeze(0).to(device)
            if layer_id is None:
                f_src = extract_dino_features(model, src_tensor)
                f_trg = extract_dino_features(model, trg_tensor)
            else:
                f_src = extract_dino_layers(model, src_tensor, [layer_id])[layer_id]
                f_trg = extract_dino_layers(model, trg_tensor, [layer_id])[layer_id]

        f_src = F.normalize(f_src, dim=1)
        f_trg = F.normalize(f_trg, dim=1)
        fh, fw = f_src.shape[2], f_src.shape[3]

        # PF-Willow uses image diagonal for normalization
        norm_factor = np.sqrt(trg_w**2 + trg_h**2)

        # Per-image counters
        image_correct = {t: 0 for t in thresholds}
        image_total_kps = 0

        # Process each keypoint pair
        for idx in range(min(len(src_kps), len(trg_kps))):
            p_src = src_kps[idx]
            p_trg = trg_kps[idx]

            # Skip invisible keypoints
            if len(p_src) < 3 or len(p_trg) < 3:
                continue
            if p_src[2] == 0 or p_trg[2] == 0:
                continue

            # Map Source Point -> Feature Grid
            feat_x = int(round(p_src[0] / src_w * (fw - 1)))
            feat_y = int(round(p_src[1] / src_h * (fh - 1)))
            feat_x = min(max(feat_x, 0), fw - 1)
            feat_y = min(max(feat_y, 0), fh - 1)

            # Get Source Descriptor
            target_feat = f_src[:, :, feat_y, feat_x]

            # Compute Similarity Map
            sim = torch.einsum('nc,nchw->nhw', target_feat, f_trg)[0]  # (H, W)

            # Get original similarity map dimensions
            h, w = sim.shape[-2], sim.shape[-1]

            if use_softargmax:
                # TODO: Add softargmax_2d function if needed
                # For now, use argmax
                flat_idx = sim.argmax()
                pred_y_idx_coarse = flat_idx // w
                pred_x_idx_coarse = flat_idx % w

                pred_x = (((pred_x_idx_coarse.item() if torch.is_tensor(pred_x_idx_coarse) else pred_x_idx_coarse) + 0.5) / w) * trg_w
                pred_y = (((pred_y_idx_coarse.item() if torch.is_tensor(pred_y_idx_coarse) else pred_y_idx_coarse) + 0.5) / h) * trg_h
            else:
                # Standard Hard Argmax
                flat_idx = sim.argmax()
                pred_y_idx_coarse = flat_idx // w
                pred_x_idx_coarse = flat_idx % w

                pred_x = (((pred_x_idx_coarse.item() if torch.is_tensor(pred_x_idx_coarse) else pred_x_idx_coarse) + 0.5) / w) * trg_w
                pred_y = (((pred_y_idx_coarse.item() if torch.is_tensor(pred_y_idx_coarse) else pred_y_idx_coarse) + 0.5) / h) * trg_h

            # Evaluate distance
            pred_x_val = pred_x.item() if torch.is_tensor(pred_x) else pred_x
            pred_y_val = pred_y.item() if torch.is_tensor(pred_y) else pred_y
            dist = np.sqrt((pred_x_val - p_trg[0])**2 + (pred_y_val - p_trg[1])**2)

            total_kps += 1
            category_total[category] += 1
            image_total_kps += 1

            for t in thresholds:
                if dist <= (t * norm_factor):
                    correct_kps[t] += 1
                    category_correct[t][category] += 1
                    image_correct[t] += 1

        # Store per-image result
        per_image_results.append({
            'category': category,
            'src_image': src_name,
            'trg_image': trg_name,
            'total_kps': image_total_kps,
            **{f'PCK@{t}': (image_correct[t] / image_total_kps * 100) if image_total_kps > 0 else 0.0
               for t in thresholds}
        })

        # Print progress every 100 pairs
        num_pairs_processed += 1
        if num_pairs_processed % 100 == 0:
            current_pck = {t: (correct_kps[t] / total_kps * 100) if total_kps > 0 else 0.0 for t in thresholds}
            print(f"\n  [{num_pairs_processed}/{len(pair_list)}] Current PCK: " +
                  " | ".join([f"@{t}={current_pck[t]:.2f}%" for t in thresholds]))

    # Compute per-keypoint results
    per_keypoint_overall = {
        f"PCK@{t}": (correct_kps[t] / total_kps * 100) if total_kps > 0 else 0.0
        for t in thresholds
    }

    per_keypoint_per_category = {
        category: {
            f"PCK@{t}": (category_correct[t][category] / category_total[category] * 100)
            if category_total[category] > 0 else 0.0
            for t in thresholds
        }
        for category in sorted(category_total.keys())
    }

    return {
        'per_keypoint': {
            'overall': per_keypoint_overall,
            'per_category': per_keypoint_per_category,
        },
        'per_image': per_image_results,
        'total_keypoints': total_kps,
        'total_images': len(per_image_results)
    }


In [ ]:
# ====================
# PRINT RESULTS FUNCTION
# ====================

def print_results(results, model_name, thresholds):
    """Print results in the same format as main.ipynb"""
    print(f"\n{'='*80}")
    print(f"{model_name} - PF-Willow Evaluation Results")
    print(f"{'='*80}")

    # Overall PCK
    print("\n1. OVERALL PER-KEYPOINT PCK")
    print("-" * 80)
    print(f"{'Threshold':<15}", end='')
    for t in thresholds:
        print(f"PCK@{t:<8}", end='  ')
    print()
    print("-" * 80)
    print(f"{'Overall':<15}", end='')
    for t in thresholds:
        pck = results['per_keypoint']['overall'][f'PCK@{t}']
        print(f"{pck:<8.2f}", end='  ')
    print()

    # Per-category PCK
    print("\n2. PER-CATEGORY PCK")
    print("-" * 80)
    print(f"{'Category':<15}", end='')
    for t in thresholds:
        print(f"PCK@{t:<8}", end='  ')
    print()
    print("-" * 80)

    for category in sorted(results['per_keypoint']['per_category'].keys()):
        print(f"{category:<15}", end='')
        for t in thresholds:
            pck = results['per_keypoint']['per_category'][category][f'PCK@{t}']
            print(f"{pck:<8.2f}", end='  ')
        print()

    print(f"\nTotal keypoints: {results['total_keypoints']}")
    print(f"Total image pairs: {results['total_images']}")

    # Save results to Google Drive
    df = pd.DataFrame(results['per_image'])
    output_dir = '/content/drive/MyDrive/AML-PROJECT-DATA/results_task4'
    output_path = os.path.join(output_dir, f"{model_name.replace(' ', '_')}_pfwillow_results.csv")
    os.makedirs(output_dir, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"\nSaved per-image results to: {output_path}")


RUN EVALUATION - CHOOSE ONE OF THE FOLLOWING CELLS TO RUN THE EVALUATION ON THE SELECTED MODEL

In [ ]:
# ====================
# RUN EVALUATIONS
# ====================
THRESHOLD = [0.05, 0.1, 0.15, 0.2]

In [19]:
# Evaluate DINOv2
print("\n Evaluating DINOv2 Finetuned ")
results_dinov2 = computePCKatT_PFWillow(
    model=dinov2_vitb14,
    dataset_root=PFWILLOW_ROOT,
    layer_id=None,
    thresholds=THRESHOLD,
    img_size=(518, 518),
    use_softargmax=False
)
print_results(results_dinov2, 'DINOv2_Finetuned', THRESHOLD)


 Evaluating DINOv2 Finetuned 
Auto-discovered 10 categories: ['motorbike(S)', 'winebottle(M)', 'car(S)', 'motorbike(G)', 'car(G)', 'car(M)', 'winebottle(wC)', 'duck(S)', 'motorbike(M)', 'winebottle(woC)']
Generating PF-Willow image pairs...
  Category 'motorbike(S)': 10 images -> 90 pairs
  Category 'winebottle(M)': 10 images -> 90 pairs
  Category 'car(S)': 10 images -> 90 pairs
  Category 'motorbike(G)': 10 images -> 90 pairs
  Category 'car(G)': 10 images -> 90 pairs
  Category 'car(M)': 10 images -> 90 pairs
  Category 'winebottle(wC)': 10 images -> 90 pairs
  Category 'duck(S)': 10 images -> 90 pairs
  Category 'motorbike(M)': 10 images -> 90 pairs
  Category 'winebottle(woC)': 10 images -> 90 pairs
Generated total of 900 image pairs across 10 categories


Evaluating pairs:   5%|▌         | 47/900 [00:10<03:16,  4.34it/s]


KeyboardInterrupt: 

In [ ]:
# Evaluate DINOv3
print("\n Evaluating DINOv3 Finetuned ")
results_dinov3 = computePCKatT_PFWillow(
    model=dinov3_vitb16,
    dataset_root=PFWILLOW_ROOT,
    layer_id=None,
    thresholds=THRESHOLD,
    img_size=(512, 512),
    use_softargmax=False
)
print_results(results_dinov3, 'DINOv3_Finetuned', THRESHOLD)

In [ ]:
# Evaluate SAM
print("\n Evaluating SAM Finetuned ")
results_sam = computePCKatT_PFWillow(
    model=sam_predictor,
    dataset_root=PFWILLOW_ROOT,
    layer_id=None,
    thresholds=THRESHOLD,
    img_size=(512, 512),
    use_softargmax=False
)
print_results(results_sam, 'SAM_Finetuned', THRESHOLD)

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)

In [ ]:
# ====================
# COMPARATIVE SUMMARY
# ====================

print("\n" + "="*80)
print("COMPARATIVE SUMMARY")
print("="*80)

summary_data = []
for model_name, results in [
    ('DINOv2_Finetuned', results_dinov2),
    ('DINOv3_Finetuned', results_dinov3),
    ('SAM_Finetuned', results_sam)
]:
    row = {'Model': model_name}
    for t in THRESHOLD:
        pck = results['per_keypoint']['overall'][f'PCK@{t}']
        row[f'PCK@{t}'] = f"{pck:.2f}"
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

# Save summary to Google Drive
output_dir = '/content/drive/MyDrive/AML-PROJECT-DATA/results_task4'
summary_path = os.path.join(output_dir, 'pfwillow_summary.csv')
os.makedirs(output_dir, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")